## 04. Modeliranje: Eksperiment A (SC + classical FE + ML)

### Uvoz biblioteka

In [17]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, f1_score

### Učitavanje podataka

In [18]:
DATA_DIR = "../data/processed"
FEATURES_DIR = "../data/features"
RESULTS_DIR = "../data/results"
MODELS_DIR = "../data/models"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

RANDOM_STATE = 42

# Train/test split (Entry + enkodirana labela) 
train_entries_df = pd.read_csv(os.path.join(FEATURES_DIR, "sc_train_entries.csv"))
test_entries_df = pd.read_csv(os.path.join(FEATURES_DIR, "sc_test_entries.csv"))

entries_sc_train = train_entries_df["Entry"].values
entries_sc_test = test_entries_df["Entry"].values

y_sc_train = train_entries_df["label_encoded"].values
y_sc_test = test_entries_df["label_encoded"].values

# LabelEncoder fit-ovan jednom u 03 (nema ponovnog fitovanja)
le = joblib.load(os.path.join(FEATURES_DIR, "sc_label_encoder.pkl"))

print(f"Train skup: {len(entries_sc_train)} proteina")
print(f"Test skup:  {len(entries_sc_test)} proteina")
print(f"Klase: {list(le.classes_)}")

Train skup: 5330 proteina
Test skup:  1333 proteina
Klase: ['Hydrolase', 'Receptor', 'Structural protein', 'Transcription factor', 'Transport protein']


### AAC-CV 

In [22]:
# AAC-CV je izračunat jednom za cijeli dataset (fiksni vokabular, ne zahtijeva fitovanje)
# Selektujemo redove koji pripadaju SC train/test skupu
aac_df = pd.read_csv(os.path.join(FEATURES_DIR, "aac_cv_features.csv"), index_col="Entry")

sc_train_aac = aac_df.loc[entries_sc_train].values
sc_test_aac = aac_df.loc[entries_sc_test].values

print(f"AAC-CV train: {sc_train_aac.shape} | test: {sc_test_aac.shape}")

AAC-CV train: (5330, 20) | test: (1333, 20)


### TF-IDF + SVD i Fizičko-hemijske osobine

In [26]:
# Fitovano na SC train skupu u 03
sc_train_tfidf = np.load(os.path.join(FEATURES_DIR, "sc_train_tfidf_svd.npy"))
sc_test_tfidf = np.load(os.path.join(FEATURES_DIR, "sc_test_tfidf_svd.npy"))

# Fizicko-hemijske osobine, fitovane na SC train skupu u 03
sc_train_ph = np.load(os.path.join(FEATURES_DIR, "sc_train_physchem_scaled.npy"))
sc_test_ph = np.load(os.path.join(FEATURES_DIR, "sc_test_physchem_scaled.npy"))

print(f"TF-IDF-SVD train: {sc_train_tfidf.shape} | test: {sc_test_tfidf.shape}")
print(f"PH train: {sc_train_ph.shape} | test: {sc_test_ph.shape}")

TF-IDF-SVD train: (5330, 150) | test: (1333, 150)
PH train: (5330, 5) | test: (1333, 5)


### Kombinovanje u feature setove

In [ ]:
feature_sets_train = {
    "AAC-CV": sc_train_aac,
    "TFIDF-SVD": sc_train_tfidf,
    "PH": sc_train_ph,
    "AAC-CV+TFIDF-SVD": np.hstack([sc_train_aac, sc_train_tfidf]),
    "AAC-CV+PH": np.hstack([sc_train_aac, sc_train_ph]),
    "TFIDF-SVD+PH": np.hstack([sc_train_tfidf, sc_train_ph]),
}

feature_sets_test = {
    "AAC-CV": sc_test_aac,
    "TFIDF-SVD": sc_test_tfidf,
    "PH": sc_test_ph,
    "AAC-CV+TFIDF-SVD": np.hstack([sc_test_aac, sc_test_tfidf]),
    "AAC-CV+PH": np.hstack([sc_test_aac, sc_test_ph]),
    "TFIDF-SVD+PH": np.hstack([sc_test_tfidf, sc_test_ph]),
}

for name, matrix in feature_sets_train.items():
    print(f"{name:<20} train: {matrix.shape} test: {feature_sets_test[name].shape}")

AAC-CV               train: (5330, 20) test: (1333, 20)
TFIDF-SVD            train: (5330, 150) test: (1333, 150)
PH                   train: (5330, 5) test: (1333, 5)
AAC-CV+TFIDF-SVD     train: (5330, 170) test: (1333, 170)
AAC-CV+PH            train: (5330, 25) test: (1333, 25)
TFIDF-SVD+PH         train: (5330, 155) test: (1333, 155)


### Definisanje modela i mreže hiperparametara

In [30]:
param_grids = {
    "LogisticRegression": {
        "model": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
        "params": {
            "C": [0.01, 0.1, 1, 10, 100],
            "solver": ["lbfgs"],
        },
    },
    "RandomForest": {
        "model": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "params": {
            "n_estimators": [200, 400],
            "max_depth": [10, 20, None],
            "min_samples_leaf": [1, 5],
        },
    },
    "SVM": {
        "model": SVC(class_weight="balanced", random_state=RANDOM_STATE),
        "params": {
            "C": [0.1, 1, 10],
            "kernel": ["rbf", "linear"],
        },
    },
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

### Glavna petlja treniranja

In [39]:
all_results = []

for feature_name, X_train in feature_sets_train.items():
    X_test = feature_sets_test[feature_name]

    for model_name, config in param_grids.items():
        print(f"Treniranje: {model_name:<20} | Feature set: {feature_name}")

        grid = GridSearchCV(
            estimator=config["model"],
            param_grid=config["params"],
            cv=cv,
            scoring="f1_weighted",
            n_jobs=-1,
            refit=True,
        )
        grid.fit(X_train, y_sc_train)

        best_model = grid.best_estimator_

        # Predikcije na train skupu (isti fit-ovan model, bez dodatnog treniranja)
        y_train_pred = best_model.predict(X_train)
        train_acc = accuracy_score(y_sc_train, y_train_pred)
        train_f1 = f1_score(y_sc_train, y_train_pred, average="weighted")

        # Predikcije na test skupu
        y_test_pred = best_model.predict(X_test)
        test_acc = accuracy_score(y_sc_test, y_test_pred)
        test_f1 = f1_score(y_sc_test, y_test_pred, average="weighted")

        all_results.append({
            "Feature set": feature_name,
            "Model": model_name,
            "Best params": grid.best_params_,
            "CV F1 (weighted)": grid.best_score_,
            "Train Accuracy": train_acc,
            "Train F1 (weighted)": train_f1,
            "Test Accuracy": test_acc,
            "Test F1 (weighted)": test_f1,
            "Overfit Gap (F1)": train_f1 - test_f1,
        })

        model_filename = f"sc_{model_name}_{feature_name}.pkl"
        joblib.dump(best_model, os.path.join(MODELS_DIR, model_filename))

print("\nTreniranje završeno.")

Treniranje: LogisticRegression   | Feature set: AAC-CV
Treniranje: RandomForest         | Feature set: AAC-CV
Treniranje: SVM                  | Feature set: AAC-CV
Treniranje: LogisticRegression   | Feature set: TFIDF-SVD
Treniranje: RandomForest         | Feature set: TFIDF-SVD
Treniranje: SVM                  | Feature set: TFIDF-SVD
Treniranje: LogisticRegression   | Feature set: PH
Treniranje: RandomForest         | Feature set: PH
Treniranje: SVM                  | Feature set: PH
Treniranje: LogisticRegression   | Feature set: AAC-CV+TFIDF-SVD
Treniranje: RandomForest         | Feature set: AAC-CV+TFIDF-SVD
Treniranje: SVM                  | Feature set: AAC-CV+TFIDF-SVD
Treniranje: LogisticRegression   | Feature set: AAC-CV+PH
Treniranje: RandomForest         | Feature set: AAC-CV+PH
Treniranje: SVM                  | Feature set: AAC-CV+PH
Treniranje: LogisticRegression   | Feature set: TFIDF-SVD+PH
Treniranje: RandomForest         | Feature set: TFIDF-SVD+PH
Treniranje: SVM  

### Pregled i čuvanje rezultata

In [40]:
results_df = pd.DataFrame(all_results)

# Sortiranje po Test F1, uz uvid u Overfit Gap
results_df = results_df.sort_values(by="Test F1 (weighted)", ascending=False).reset_index(drop=True)
results_df.to_csv(os.path.join(RESULTS_DIR, "sc_classical_modeling_results.csv"), index=False)

display_cols = [
    "Model", "Feature set",
    "Train Accuracy", "Train F1 (weighted)",
    "CV F1 (weighted)",
    "Test Accuracy", "Test F1 (weighted)",
    "Overfit Gap (F1)",
]
results_df[display_cols].style.format({
    "Train Accuracy": "{:.3f}",
    "Train F1 (weighted)": "{:.3f}",
    "CV F1 (weighted)": "{:.3f}",
    "Test Accuracy": "{:.3f}",
    "Test F1 (weighted)": "{:.3f}",
    "Overfit Gap (F1)": "{:.3f}",
}).background_gradient(subset=["Overfit Gap (F1)"], cmap="Reds")

,Model,Feature set,Train Accuracy,Train F1 (weighted),CV F1 (weighted),Test Accuracy,Test F1 (weighted),Overfit Gap (F1)
0,SVM,AAC-CV+TFIDF-SVD,0.988,0.988,0.830,0.818,0.816,0.172
1,SVM,TFIDF-SVD,0.992,0.992,0.829,0.812,0.809,0.183
2,SVM,TFIDF-SVD+PH,0.911,0.911,0.804,0.796,0.795,0.116
3,RandomForest,AAC-CV+PH,1.000,1.000,0.764,0.764,0.760,0.240
4,SVM,AAC-CV,0.859,0.858,0.763,0.759,0.759,0.100
5,LogisticRegression,TFIDF-SVD+PH,0.795,0.797,0.764,0.756,0.758,0.039
6,RandomForest,TFIDF-SVD+PH,0.982,0.982,0.768,0.760,0.756,0.226
7,RandomForest,AAC-CV+TFIDF-SVD,0.983,0.983,0.765,0.760,0.755,0.227
8,LogisticRegression,AAC-CV+TFIDF-SVD,0.784,0.786,0.753,0.744,0.747,0.039
9,RandomForest,AAC-CV,0.953,0.953,0.746,0.746,0.743,0.209


### Classification report za najbolju kombinaciju

In [41]:
best_row = results_df.iloc[0]
print(f"Najbolja kombinacija: {best_row['Model']} + {best_row['Feature set']}")
print(f"Parametri: {best_row['Best params']}\n")

best_model_path = os.path.join(MODELS_DIR, f"sc_{best_row['Model']}_{best_row['Feature set']}.pkl")
best_model = joblib.load(best_model_path)
best_X_test = feature_sets_test[best_row["Feature set"]]

y_pred_best = best_model.predict(best_X_test)
print(classification_report(y_sc_test, y_pred_best, target_names=le.classes_))

Najbolja kombinacija: SVM + AAC-CV+TFIDF-SVD
Parametri: {'C': 10, 'kernel': 'rbf'}

                      precision    recall  f1-score   support

           Hydrolase       0.78      0.87      0.82       441
            Receptor       0.86      0.86      0.86       276
  Structural protein       0.80      0.61      0.69       142
Transcription factor       0.90      0.92      0.91       269
   Transport protein       0.76      0.67      0.71       205

            accuracy                           0.82      1333
           macro avg       0.82      0.78      0.80      1333
        weighted avg       0.82      0.82      0.82      1333

